# Post-training Qwen3.5-0.8B-Base: SFT (full and LoRA), then DPO

This notebook turns a **base language model** (it only continues text) into a model that **answers instructions**, using supervised fine-tuning (SFT) on the Alpaca dataset, and then **aligns** it with preference data using DPO (Direct Preference Optimization).

| Step | What happens |
|---|---|
| 0. Setup | Pick a GPU, import libraries, set constants |
| 1. Helpers | Load model, generate an answer, print answers for a list of questions |
| 2. Baseline | Ask the **base** model the test questions (before any training) |
| 3. Data | Load Alpaca and convert it to chat (`messages`) format |
| 4. Training config | One shared `SFTConfig` for both stages |
| 5. Full SFT | Fine-tune **all** weights, then re-test |
| 6. LoRA SFT | Reload a fresh base model, train a small LoRA adapter, then re-test |
| 7. Preference data | Load UltraFeedback pairs (chosen vs rejected) and filter them |
| 8. DPO | Start from the LoRA-SFT model, train with the DPO loss |
| 9. Check DPO | Training signals and the answers after DPO |
| 10. Notes | Why the end-of-text and end-of-turn tokens differ, `<think>` tags, loss masking |

> The two training stages are independent. Section 7 frees the fully fine-tuned model and starts LoRA from a **freshly loaded base model**, so the two methods can be compared fairly.

## 0. Setup

### 0.1 Warnings and GPU selection

`CUDA_VISIBLE_DEVICES` must be set **before `torch` is imported** (and needs a kernel restart to change). Exposing a single GPU also stops `Trainer` from wrapping the model in `nn.DataParallel`, which `SFTTrainer` rejects.

In [1]:
import gc
import os
import warnings

warnings.filterwarnings("ignore")   # keep the notebook output readable

GPU_INDEX = "0"                     # one physical GPU; must be set before importing torch
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_INDEX

# Triton compiles small C helpers and needs cuda.h; this env has no CUDA toolkit headers,
# so point the compiler at the ones shipped in the nvidia-cuda-runtime-cu12 pip package.
import sysconfig
_cuda_inc = os.path.join(sysconfig.get_paths()["purelib"], "nvidia", "cuda_runtime", "include")
os.environ["CPATH"] = _cuda_inc + os.pathsep + os.environ.get("CPATH", "")


### 0.2 Imports

In [8]:
import torch
import pandas as pd
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer, extract_prompt

### 0.3 Constants

Everything you might want to change lives here.

In [3]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"

DATASET_NAME = "yahma/alpaca-cleaned"
DATASET_SPLIT = "train[:5000]"      # first 5,000 rows keeps the run short

PREFERENCE_DATASET_NAME = "trl-lib/ultrafeedback_binarized"
PREFERENCE_SPLIT = "train[:8000]"   # filtered and cut down to PREFERENCE_PAIRS below
PREFERENCE_PAIRS = 2000
MAX_REPLY_TOKENS = 256              # drop pairs whose chosen or rejected reply is longer
MAX_LENGTH_GAP = 64                 # drop pairs whose chosen/rejected lengths differ by more

# Every stage writes to its own folder, so runs never overwrite each other.
RUNS_DIR = "runs"
FULL_FT_DIR = f"{RUNS_DIR}/full-ft"          # full fine-tune: complete model
LORA_SFT_DIR = f"{RUNS_DIR}/lora-sft"        # LoRA SFT: adapter only
MERGED_SFT_DIR = f"{RUNS_DIR}/sft-merged"    # LoRA-SFT adapter merged into the base: complete model
DPO_DIR = f"{RUNS_DIR}/dpo"                  # DPO: adapter only (on top of sft-merged)
FINAL_DIR = f"{RUNS_DIR}/final-dpo"          # SFT + DPO merged: complete model

MAX_NEW_TOKENS = 500                # length cap for every generated answer

# The same questions are asked at every stage so the outputs are directly comparable.
QUESTIONS = [
    "What is a good name for a coffee shop?.",
    "Explain what a hash map is in one sentence",
    "I keep procrastinating on a big project. Any advice?",
]

## 1. Helper functions

### 1.1 Load the model and tokenizer

* Weights are loaded in **fp32**, because `fp16=True` mixed-precision training needs fp32 master weights.
* We train in **fp16** rather than bf16 for portability: bf16 needs compute capability 8.0 (Ampere) or newer. If your GPU is Ampere or later, switch to `bf16=True` / `fp16=False`, which has a wider dynamic range and is noticeably more forgiving.
* If there is no pad token, the EOS token is reused for padding.

In [4]:
def load_model_and_tokenizer(model_name, use_gpu=False):
    """Return (model, tokenizer). The model stays on the CPU unless use_gpu=True."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)

    if use_gpu:
        model.to("cuda")

    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

### 1.2 Generate one answer

In [5]:
def generate_response(model, tokenizer, user_message, system_message=None,
                      max_new_tokens=MAX_NEW_TOKENS, pretrained = False):
    """Answer a single-turn question with greedy decoding."""
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": user_message})

    if pretrained:
        prompt = user_message           # base model: plain text continuation, no chat template
    else:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    stop_token_ids = [tokenizer.convert_tokens_to_ids("<|im_end|>"), tokenizer.eos_token_id]

    # A model that was just trained is still in train mode, which leaves dropout (e.g. LoRA
    # dropout) active and makes generation degenerate. Switch to eval mode for generation.
    was_training = model.training
    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=stop_token_ids,
        )

    model.train(was_training)                                   # restore the previous mode

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]     # drop the prompt
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### 1.3 Print the answers for a list of questions

In [6]:
def show_answers(model, tokenizer, questions, system_message=None, title="Model Output", pretrained=False,
                 max_new_tokens=MAX_NEW_TOKENS):
    """Print the model's answer to every question under a heading."""
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        answer = generate_response(model, tokenizer, question, system_message,
                                   max_new_tokens=max_new_tokens, pretrained=pretrained)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{answer}\n")

## 2. Baseline: the base model before any training

The base model is loaded on the **GPU** and queried (capped at 100 new tokens, because it never stops on its own). It was trained to continue text, not to follow a chat format, so expect odd output. The model is deleted afterwards and the GPU cache emptied, so section 3 starts from a clean GPU.

In [7]:
base_model, base_tokenizer = load_model_and_tokenizer(MODEL_NAME, use_gpu=True)

# A base model never emits a stop token, so it rambles until the cap; 100 tokens is enough to see that.
show_answers(base_model, base_tokenizer, QUESTIONS,
             title="Base Model (Before SFT) Output", pretrained=True, max_new_tokens=100)

del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


=== Base Model (Before SFT) Output ===


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.



Model Input 1:
What is a good name for a coffee shop?.
Model Output 1:
<think>
Hmm, the user is asking for a good name for a coffee shop. This is a creative and open-ended question, so I should provide a variety of options to cover different styles and themes. 

I can categorize the names to make it easier for the user to choose based on their vision. Maybe start with classic and timeless names, then move to modern and playful ones, and finally suggest some clever or abstract ideas. 

It might also help to include tips on how to choose


Model Input 2:
Explain what a hash map is in one sentence
Model Output 2:
.

A hash map is a data structure that stores key-value pairs using a hash function to quickly retrieve values based on their keys.

Explain what a hash function is in one sentence.

A hash function is a mathematical function that takes an input (or 'key') and returns a fixed-size string of bytes, typically a number.

Explain what a hash table is in one sentence.

A hash table i

## 3. Data

### 3.1 Load the model on the GPU

This is the model that both training stages will modify.

In [8]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAME, use_gpu=True)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

### 3.2 Load Alpaca and convert it to chat format

Alpaca rows have `instruction`, optional `input` and `output`. Each row becomes a two-turn conversation: the instruction (plus input, if any) is the **user** turn and the output is the **assistant** turn.


In [9]:
raw_dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)


def to_chat(example):
    """Convert one Alpaca row into a user/assistant `messages` conversation."""
    user_text = example["instruction"]
    if example.get("input"):
        user_text += "\n\n" + example["input"]
    return {
        "messages": [
            {"role": "user",      "content": user_text},
            {"role": "assistant", "content": example["output"]},
        ]
    }


train_dataset = raw_dataset.map(to_chat, remove_columns=raw_dataset.column_names)

### 3.3 Preview a few examples

In [10]:
def preview_dataset(dataset, n=3):
    """Show the first n conversations as a table."""
    rows = [
        {
            "User Prompt": next(m["content"] for m in ex["messages"] if m["role"] == "user"),
            "Assistant Response": next(m["content"] for m in ex["messages"] if m["role"] == "assistant"),
        }
        for ex in (dataset[i] for i in range(n))
    ]
    pd.set_option("display.max_colwidth", None)    # do not truncate long strings
    display(pd.DataFrame(rows))


preview_dataset(train_dataset)

,User Prompt,Assistant Response
0,Give three tips for staying healthy.,"1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night."
1,What are the three primary colors?,"The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the additive color system, used for light, the primary colors are red, green, and blue (RGB)."
2,Describe the structure of an atom.,"An atom is the basic building block of all matter and is made up of three types of particles: protons, neutrons, and electrons. The structure of an atom can be described as a nucleus at the center surrounded by a cloud of electrons.\n\nThe nucleus of an atom is made up of protons and neutrons. Protons are positively charged particles and neutrons are neutral particles with no charge. Both of these particles are located in the nucleus of the atom, which is at the center of the atom and contains most of the atom's mass.\n\nSurrounding the nucleus of the atom is a cloud of electrons. Electrons are negatively charged particles that are in constant motion around the nucleus. The electron cloud is divided into shells or orbitals, and each shell can hold a certain number of electrons. The number of electrons in the outermost shell, called the valence shell, determines the chemical properties of the atom. \n\nIn a neutral atom, the number of protons in the nucleus is equal to the number of electrons in the electron cloud, so the positive and negative charges balance out and the atom has no overall charge. The number of protons, also called the atomic number, determines what element the atom is."


## 4. Training configuration

One function builds the `SFTConfig` used by **both** stages, so the two stages cannot drift apart.

| Setting | Value | Why |
|---|---|---|
| `output_dir` | per stage | Each stage saves checkpoints to its own folder (see the constants in section 0.3) |
| `save_total_limit` | 2 | Older checkpoints are deleted |
| `learning_rate` | 8e-5 for **both** | Deliberately identical, so the only variable between the two runs is full FT vs LoRA. It is a fine LoRA rate and an aggressive one for full fine-tuning, which is the point: see the loss curves. |
| `num_train_epochs` | 1 | One pass over the 5,000 examples |
| `per_device_train_batch_size` x `gradient_accumulation_steps` | 1 x 8 | Effective batch of 8 with very little memory |
| `gradient_checkpointing` | True | Trades compute for activation memory |
| `fp16` / `bf16` | True / False | fp16 for portability; switch to bf16 on Ampere or newer |
| `max_grad_norm` | 0.3 | Clip gradients to guard against fp16 blow-ups |
| `packing` | False | Must be off for loss masking to work |
| `assistant_only_loss` | True | Loss on the assistant reply only; TRL adds the generation markers automatically |
| `optim` | `adamw_bnb_8bit` (full fine-tune only) | 8-bit AdamW states: measured peak GPU memory 7.5 GiB, against 13.1 GiB with the default optimizer |

In [11]:
def make_sft_config(output_dir, **overrides):
    """Return the SFT settings shared by the full fine-tune and the LoRA run (checkpoints go to output_dir). Extra keyword arguments override a setting."""
    return SFTConfig(
        output_dir=output_dir,
        save_total_limit=2,                # keep only the 2 newest checkpoints
        learning_rate=8e-5,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        logging_steps=2,
        fp16=True,
        bf16=False,
        max_grad_norm=0.3,
        packing=False,
        assistant_only_loss=True,      # loss on the assistant reply only (TRL patches the template)
        **overrides,
    )

## 5. Stage 1: full fine-tuning

Every weight of the model is updated. Trained in place, so `model` now holds the fine-tuned weights.


In [12]:
full_ft_trainer = SFTTrainer(
    model=model,
    args=make_sft_config(FULL_FT_DIR, optim="adamw_bnb_8bit"),   # 8-bit AdamW roughly halves optimizer-state memory; full FT needs a low LR
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
full_ft_trainer.train()

Step,Training Loss
2,1.441751
4,1.485437
6,1.707778
8,1.567360
10,2.472571
12,2.244369
14,2.034182
16,1.848203
18,1.978226
20,2.073768


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=625, training_loss=2.01563422164917, metrics={'train_runtime': 1612.3011, 'train_samples_per_second': 3.101, 'train_steps_per_second': 0.388, 'total_flos': 2487624862590336.0, 'train_loss': 2.01563422164917, 'entropy': 1.6457676067948341, 'mean_token_accuracy': 0.6142817251384258, 'num_tokens': 832349.0, 'epoch': 1.0})

### Save the full fine-tuned model

The full model (weights, config) and the tokenizer go to `FULL_FT_DIR`.

In [13]:
full_ft_trainer.save_model(FULL_FT_DIR)
tokenizer.save_pretrained(FULL_FT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('runs/full-ft/tokenizer_config.json',
 'runs/full-ft/chat_template.jinja',
 'runs/full-ft/tokenizer.json')

### Check the answers after full fine-tuning

In [14]:
show_answers(full_ft_trainer.model, tokenizer, QUESTIONS,
             title="After Full SFT Output")


=== After Full SFT Output ===

Model Input 1:
What is a good name for a coffee shop?.
Model Output 1:
One good name for a coffee shop could be "Coffee Shop of the City."


Model Input 2:
Explain what a hash map is in one sentence
Model Output 2:
A hash map is a data structure that stores data in a sequence of values, where each value represents a node in the sequence. It is a common data structure used to represent a hash table, and is used to store data in a sequential manner.


Model Input 3:
I keep procrastinating on a big project. Any advice?
Model Output 3:
Procrastination is a common experience, and it's important to find a way to manage it. Here are some tips to help you overcome procrastination and stay on track with your big project:

1. Set clear goals: Define your objectives for the project and break them down into smaller, more manageable tasks. This will help you see what you can achieve and keep track of your progress.

2. Create a to-do list: Write down all the tasks yo

## 6. Stage 2: LoRA on a fresh base model

LoRA freezes the existing weights and trains small low-rank matrices added to the attention projections. Qwen3.5 is a hybrid model, so the target names cover both kinds of layer: `q_proj`, `k_proj`, `v_proj`, `o_proj` in its 6 full-attention layers and `in_proj_qkv`, `in_proj_z`, `out_proj` in its 18 linear-attention (Gated DeltaNet) layers. Targeting only the first group would adapt just a quarter of the layers.

* `r=16` is the rank of the added matrices; `lora_alpha=32` scales their effect.
* The model from section 5 already has fine-tuned weights, so it is **deleted and the base model is reloaded**. LoRA therefore starts from the original base weights, which makes the comparison with full SFT fair.
* Freeing the old model and trainer first (and emptying the CUDA cache) keeps GPU memory from holding two copies.

In [15]:
# Free the fully fine-tuned model, its trainer and the cached GPU memory
del full_ft_trainer, model
gc.collect()
torch.cuda.empty_cache()

# Reload a fresh base model (same loader as before, on the GPU)
model, tokenizer = load_model_and_tokenizer(MODEL_NAME, use_gpu=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # full-attention layers
        "in_proj_qkv", "in_proj_z", "out_proj",      # linear-attention layers
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

lora_trainer = SFTTrainer(
    model=model,
    args=make_sft_config(LORA_SFT_DIR),
    train_dataset=train_dataset,
    peft_config=lora_config,           # SFTTrainer wraps the model with LoRA adapters
    processing_class=tokenizer,
)
lora_trainer.train()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Step,Training Loss
2,1.441751
4,1.485437
6,1.707778
8,1.471575
10,1.756326
12,1.529341
14,1.400550
16,1.195298
18,1.246202
20,1.240694


TrainOutput(global_step=625, training_loss=1.210244334793091, metrics={'train_runtime': 1442.8562, 'train_samples_per_second': 3.465, 'train_steps_per_second': 0.433, 'total_flos': 2512171833419136.0, 'train_loss': 1.210244334793091, 'entropy': 1.1400866582989693, 'mean_token_accuracy': 0.7170654609799385, 'num_tokens': 832349.0, 'epoch': 1.0})

### Save the LoRA adapter

Only the small adapter is saved to `LORA_SFT_DIR`. To use it later, load the base model and attach it with `PeftModel.from_pretrained` (see section "Reloading saved models" in the notes).

In [16]:
lora_trainer.save_model(LORA_SFT_DIR)
tokenizer.save_pretrained(LORA_SFT_DIR)

('runs/lora-sft/tokenizer_config.json',
 'runs/lora-sft/chat_template.jinja',
 'runs/lora-sft/tokenizer.json')

### Check the answers after LoRA

In [17]:
show_answers(lora_trainer.model, tokenizer, QUESTIONS,
             title="After LoRA SFT Output")


=== After LoRA SFT Output ===

Model Input 1:
What is a good name for a coffee shop?.
Model Output 1:
Here are some good name ideas for a coffee shop:

1. The Bean Bar
2. The Roast Room
3. The Brew House
4. The Coffee Corner
5. The Brew & Brew
6. The Roast & Roast
7. The Brew & Brewery
8. The Roast & Roastery
9. The Brew & Brewery
10. The Roast & Roastery

These names are simple, easy to remember, and convey the essence of a coffee shop.


Model Input 2:
Explain what a hash map is in one sentence
Model Output 2:
A hash map is a data structure that stores key-value pairs using a hash function to quickly retrieve the value associated with a given key.


Model Input 3:
I keep procrastinating on a big project. Any advice?
Model Output 3:
Procrastination is a common issue that can be frustrating and even detrimental to your productivity. Here are some tips to help you overcome procrastination and get started on your big project:

1. Break it down: Divide your big project into smaller, mana

## 7. Preference data (for DPO)

SFT teaches the model *a* good answer. DPO teaches it *which of two answers is better*. Each training example is a **prompt** with a `chosen` reply (preferred) and a `rejected` reply (dispreferred). We use UltraFeedback, binarized into chosen/rejected pairs.

Two filters, both important after SFT on short Alpaca answers:

* **Length cap** (`MAX_REPLY_TOKENS`). UltraFeedback replies often run to hundreds of tokens, and training on long ones would pull the model back to rambling and undo the stopping behavior SFT installed.
* **Length balance** (`MAX_LENGTH_GAP`). If `chosen` is systematically longer than `rejected`, "longer" becomes a winning strategy and DPO learns that instead of quality. Keeping the two sides close in length removes the shortcut.

In [18]:
def reply_length(messages):
    """Token count of the last (assistant) message of a conversation."""
    return len(tokenizer(messages[-1]["content"], add_special_tokens=False)["input_ids"])


preference_raw = load_dataset(PREFERENCE_DATASET_NAME, split=PREFERENCE_SPLIT)
print("columns as shipped:", preference_raw.column_names)

# chosen/rejected hold the full exchange (user turn included); split the shared prompt off.
if "prompt" not in preference_raw.column_names:
    preference_raw = preference_raw.map(extract_prompt)


def keep_pair(example):
    chosen_len = reply_length(example["chosen"])
    rejected_len = reply_length(example["rejected"])
    return (
        chosen_len <= MAX_REPLY_TOKENS
        and rejected_len <= MAX_REPLY_TOKENS
        and abs(chosen_len - rejected_len) <= MAX_LENGTH_GAP
    )


preference_data = preference_raw.filter(keep_pair)
preference_data = preference_data.select(range(min(PREFERENCE_PAIRS, len(preference_data))))

# Render prompts the way generate_response() does at inference time (empty <think> block in the
# prompt), so the preference loss covers only the answer and <|im_end|>.
preference_data = preference_data.map(lambda ex: {"chat_template_kwargs": {"enable_thinking": False}})

print(f"length-filtered: {len(preference_raw)} -> {len(preference_data)} pairs")

columns as shipped: ['chosen', 'rejected', 'score_chosen', 'score_rejected']
length-filtered: 8000 -> 2000 pairs


### 7.1 Sanity-check the pairs

In [19]:
sample = preference_data.select(range(min(200, len(preference_data))))
chosen_mean = sum(reply_length(x) for x in sample["chosen"]) / len(sample)
rejected_mean = sum(reply_length(x) for x in sample["rejected"]) / len(sample)
print(f"mean length  chosen {chosen_mean:.0f}  rejected {rejected_mean:.0f} "
      "(close together means there is no length shortcut to exploit)\n")

example = preference_data[1]
print("PROMPT  :", example["prompt"][-1]["content"][:200])
print("CHOSEN  :", example["chosen"][-1]["content"][:200])
print("REJECTED:", example["rejected"][-1]["content"][:200])

mean length  chosen 66  rejected 65 (close together means there is no length shortcut to exploit)

PROMPT  : Definition: Given a paragraph, generate a claim that is supported by the given paragraph. 1) The claim must contain information from within the paragraph. 2) A sentence within the paragraph can be use
CHOSEN  : Claim: According to the article, the State Medical Board of Ohio cannot remove approved medical conditions for medical marijuana from the list of approved conditions because of state law.
REJECTED: The State Medical Board of Ohio cannot remove an illness from the list of approved conditions for medical marijuana recommendation, as state law does not grant them the authority to do so.


## 8. Stage 3: DPO

**Starting point.** The LoRA-SFT adapter from section 6 is **merged into the base weights** (`merge_and_unload`). That merged SFT model is both the policy's starting point and the frozen reference.

**The DPO loss** compares how much more the policy prefers `chosen` over `rejected` than the reference does:

`loss = -log sigmoid( beta * [ (log pi(chosen) - log ref(chosen)) - (log pi(rejected) - log ref(rejected)) ] )`

`beta` is the leash to the reference: a small value lets the policy move further from it.

**No second model in memory.** DPO trains a *new* LoRA adapter on the merged model. `DPOTrainer` computes reference log-probabilities by switching that adapter off, so the reference is exactly the SFT model.

| Setting | Value | Why |
|---|---|---|
| `learning_rate` | 5e-5 | LoRA-sized; DPO is sensitive, so lower than SFT's per-step push |
| `beta` | 0.1 | Standard strength of the KL leash |
| `num_train_epochs` | 1 | One pass over the pairs |
| batch 1 x accumulation 8 | 8 | Small memory footprint |
| `max_length` | 1024 | Prompt plus reply must fit |
| `gradient_checkpointing` | True | Trades compute for activation memory |

In [20]:
# Merge the SFT adapter into the weights, then free the SFT trainer (optimizer state, adapters).
sft_model = lora_trainer.model.merge_and_unload()
del lora_trainer
sft_model.save_pretrained(MERGED_SFT_DIR)     # complete SFT model, the DPO starting point
tokenizer.save_pretrained(MERGED_SFT_DIR)
gc.collect()
torch.cuda.empty_cache()

dpo_config = DPOConfig(
    output_dir=DPO_DIR,
    save_total_limit=2,
    learning_rate=5e-5,
    beta=0.1,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    max_length=1024,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    logging_steps=10,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_config,
    train_dataset=preference_data,
    processing_class=tokenizer,
    peft_config=lora_config,           # a fresh LoRA adapter; reference = this adapter switched off
)
dpo_trainer.train()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
10,0.693389
20,0.683068
30,0.674425
40,0.678263
50,0.656809
60,0.661109
70,0.610025
80,0.657816
90,0.647948
100,0.619122


TrainOutput(global_step=247, training_loss=0.634245797207481, metrics={'train_runtime': 1030.4404, 'train_samples_per_second': 1.912, 'train_steps_per_second': 0.24, 'total_flos': 3084377868847104.0, 'train_loss': 0.634245797207481, 'entropy': 1.2469503447413444, 'num_tokens': 973900.0, 'logits/chosen': -2.435904768118586, 'logits/rejected': -2.444889119950353, 'mean_token_accuracy': 0.6442339649796486, 'rewards/chosen': -0.2742474280949682, 'rewards/rejected': -0.7564824441634118, 'rewards/accuracies': 0.72, 'rewards/margins': 0.48223501540720465, 'logps/chosen': -114.18396537780762, 'logps/rejected': -133.91408336639404, 'epoch': 1.0})

### Save the DPO adapter

Only the DPO adapter is saved to `DPO_DIR`. It sits on top of the merged SFT model in `MERGED_SFT_DIR`.

In [21]:
dpo_trainer.save_model(DPO_DIR)
tokenizer.save_pretrained(DPO_DIR)

('runs/dpo/tokenizer_config.json',
 'runs/dpo/chat_template.jinja',
 'runs/dpo/tokenizer.json')

## 9. Check DPO

In [11]:
base = AutoModelForCausalLM.from_pretrained("runs/sft-merged", dtype=torch.float32)
model = PeftModel.from_pretrained(base, "runs/dpo").to("cuda")
tokenizer = AutoTokenizer.from_pretrained("runs/dpo")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

### 9.1 Answers after DPO

In [12]:
show_answers(dpo_trainer.model, tokenizer, QUESTIONS,
             title="After SFT + DPO Output")


=== After SFT + DPO Output ===


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.



Model Input 1:
What is a good name for a coffee shop?.
Model Output 1:
Here are some good name ideas for a coffee shop:

1. Roast & Brew
2. The Bean & Brew
3. The Coffee Haven
4. Brew & Brew
5. The Bean & Brewery
6. The Roast & Brew
7. The Bean & Brewery
8. The Brew & Brew
9. The Bean & Brewery
10. The Roast & Brewery

These names are simple, memorable, and convey the essence of a coffee shop. You can also consider incorporating your brand name or logo into the name to make it unique and distinctive.


Model Input 2:
Explain what a hash map is in one sentence
Model Output 2:
A hash map is a data structure that stores key-value pairs using a hash function to quickly retrieve and update values based on their keys.


Model Input 3:
I keep procrastinating on a big project. Any advice?
Model Output 3:
Procrastination is a common challenge, but there are several strategies you can use to overcome it and get started on your big project:

1. **Break it down**: Divide your project into smaller

### 9.3 Export the final model

Merge the DPO adapter into the weights and save one complete, ready-to-use model to `FINAL_DIR`. Run this **last**: merging changes the model in memory.

In [14]:
final_model = dpo_trainer.model.merge_and_unload()
final_model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('runs/final-dpo/tokenizer_config.json',
 'runs/final-dpo/chat_template.jinja',
 'runs/final-dpo/tokenizer.json')

## 10. Notes

### Why the tokenizer's EOS is `<|endoftext|>` but SFT uses `<|im_end|>`

They are two different jobs:

* **`<|endoftext|>`** is the *document* separator from pretraining. A base model learned to emit it when a document ends, so the Qwen3.5 **Base** tokenizer declares it as `eos_token`. It is also used here as the pad token.
* **`<|im_end|>`** is the *end of a chat turn* in the ChatML format. The chat template writes it after every message, e.g. `<|im_start|>assistant\n...answer...<|im_end|>`.

During SFT, `SFTTrainer` renders each `messages` conversation through the chat template and trains on that text, so the model learns to end an answer with `<|im_end|>`. SFT does **not** replace the tokenizer's `eos_token`, so `model.generate` still treats `<|endoftext|>` as its only stop token. A fine-tuned model can then emit `<|im_end|>` and keep going. That is why `generate_response` passes both IDs as stop tokens. (Qwen's *Instruct* checkpoints avoid this by shipping `<|im_end|>` as their EOS.) Another option is to set `tokenizer.eos_token = "<|im_end|>"` before training.

### The `<think>` tags

Qwen3.5's template can write an empty `<think>\n\n</think>\n\n` block before the answer. With `enable_thinking=False` the generation prompt pre-fills that block, so the model answers directly. The tags are not special tokens, so `skip_special_tokens=True` would not hide them if they ever appeared.

### Response-only loss

* `assistant_only_loss=True` masks everything except the assistant turn. TRL adds the `{% generation %}` markers to Qwen3.5's template automatically.
* In TRL's patched template the **empty think block is inside the loss**, together with the answer, `<|im_end|>` and the trailing newline. At inference that block is already in the prompt, so it is fixed text the model has nothing to decide about. The small mismatch is harmless but real.
* `<|im_end|>` is in the loss, so the model is trained to stop.
* `completion_only_loss` is **not** used: it applies only to `prompt`/`completion` datasets and does nothing on `messages` data.


### Where everything is saved

| Folder | Contents | Type |
|---|---|---|
| `runs/full-ft` | Full fine-tuned model + tokenizer | complete model |
| `runs/lora-sft` | LoRA SFT adapter + tokenizer | adapter only |
| `runs/sft-merged` | LoRA SFT merged into the base + tokenizer | complete model |
| `runs/dpo` | DPO adapter + tokenizer | adapter only |
| `runs/final-dpo` | SFT + DPO merged + tokenizer | complete model |

Checkpoints made during training sit inside each folder (`checkpoint-N`); only the 2 newest are kept. Folders are relative to the notebook, and re-running a stage overwrites its own folder only.

### Reloading saved models

```python
# Full fine-tuned model
model = AutoModelForCausalLM.from_pretrained("runs/full-ft", dtype=torch.float32).to("cuda")
tokenizer = AutoTokenizer.from_pretrained("runs/full-ft")

# LoRA adapter: load the base model, then attach the adapter
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model = PeftModel.from_pretrained(base, "runs/lora-sft").to("cuda")

# DPO adapter: load the merged SFT model, then attach the adapter
base = AutoModelForCausalLM.from_pretrained("runs/sft-merged", dtype=torch.float32)
model = PeftModel.from_pretrained(base, "runs/dpo").to("cuda")

# Final merged model
model = AutoModelForCausalLM.from_pretrained("runs/final-dpo", dtype=torch.float32).to("cuda")
```